# 🏦 Banking Intent Classification — LLaMA 3.2-1B + Unsloth

**Project 2 — NLP Industry | HCMUS**

> Fine-tuning LLaMA-3.2-1B-Instruct trên BANKING77 (77 intents)

**Hướng dẫn**: Chạy từng cell theo thứ tự từ trên xuống dưới.

⚠️ Đảm bảo đã chọn **Runtime → Change runtime type → T4 GPU**

## ✅ Bước 0: Kiểm tra GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi --query-gpu=memory.total,memory.free --format=csv

## 📦 Bước 1: Cài đặt thư viện

In [ ]:
%%capture
# Cài Unsloth (phải cài trước transformers)
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --no-deps 'xformers<0.0.27' 'trl<0.9.0' peft accelerate bitsandbytes

# Các thư viện còn lại
!pip install datasets scikit-learn pandas pyyaml

print('✅ Cài đặt hoàn tất!')

## 📂 Bước 2: Clone repo từ GitHub

In [ ]:
# Thay YOUR_USERNAME bằng username GitHub của bạn
GITHUB_USERNAME = 'YOUR_USERNAME'
REPO_NAME = 'banking-intent-unsloth'

!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}
!ls -la

## 🗃️ Bước 3: Tiền xử lý dữ liệu BANKING77

In [ ]:
!python scripts/preprocess_data.py --config configs/train.yaml

In [ ]:
# Kiểm tra dữ liệu
import pandas as pd, json

df_train = pd.read_csv('sample_data/train.csv')
df_test  = pd.read_csv('sample_data/test.csv')

print(f'Train: {len(df_train)} mẫu')
print(f'Test : {len(df_test)} mẫu')
print(f'\nPhân phối intents (top 5):')
print(df_train['intent'].value_counts().head())
print(f'\nVí dụ:')
print(df_train[['text', 'intent']].head(3).to_string())

## 🏋️ Bước 4: Fine-tuning với Unsloth

In [ ]:
# ⏱️ Ước tính thời gian: ~45-60 phút trên T4 với 3 epochs
!python scripts/train.py --config configs/train.yaml

## 💾 Bước 5: Backup checkpoint lên Google Drive (khuyến nghị)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/banking77-checkpoint'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy checkpoint
src = 'outputs/llama32-banking77/checkpoint-final'
shutil.copytree(src, DRIVE_DIR + '/checkpoint-final', dirs_exist_ok=True)
shutil.copytree('sample_data', DRIVE_DIR + '/sample_data', dirs_exist_ok=True)

print(f'✅ Đã backup lên: {DRIVE_DIR}')

## 🔍 Bước 6: Inference — Demo kết quả

In [ ]:
import sys
sys.path.insert(0, '.')
from scripts.inference import IntentClassification

# Tải model
classifier = IntentClassification('configs/inference.yaml')

In [ ]:
# ── Demo với nhiều ví dụ ──────────────────────────────────────────────────
test_messages = [
    'I lost my credit card',
    'What is my current balance?',
    'I want to transfer money to another account',
    'My card payment was declined',
    'How do I change my PIN?',
    'I was charged twice for the same transaction',
    'What is the exchange rate for USD to EUR?',
    'I want to cancel my card',
    'When will my refund arrive?',
    'How do I activate my new card?',
]

print('=' * 60)
print('  BANKING INTENT CLASSIFICATION — DEMO')
print('=' * 60)
for msg in test_messages:
    label = classifier(msg)
    print(f'\nInput  : {msg}')
    print(f'Intent : {label}')
print('=' * 60)

In [ ]:
# ── Thử với câu của bạn ───────────────────────────────────────────────────
your_message = 'I need help with my online banking password'  # ← thay đổi ở đây

label = classifier(your_message)
print(f'Message: {your_message}')
print(f'Intent : {label}')

## 📊 Bước 7: Xem kết quả đánh giá

In [ ]:
# Xem accuracy và classification report
with open('outputs/llama32-banking77/eval_results.txt') as f:
    print(f.read()[:3000])  # In 3000 ký tự đầu